# Task 2.2 — Interestingness Score Implementation

**Paper:** Goorha, S. & Ungar, L. (2010). *Discovery of Significant Emerging Trends.* ACM SIGKDD.

---

This notebook implements the core scoring function from the paper. Every code block is preceded by a markdown annotation referencing the relevant section of the paper.

## Section 2 Reference: The Interestingness Score

In **Section 2** of the paper, Goorha & Ungar define the *interestingness score* for a phrase $p$ co-occurring with entity $e$ as:

$$I(p, e) = \frac{\text{count}(p, e)}{\text{total}(e)^{\alpha}}$$

where:
- $\text{count}(p, e)$ is the number of times phrase $p$ appears within a 100-character window of entity $e$
- $\text{total}(e)$ is the total number of phrase co-occurrences with entity $e$ across the corpus
- $\alpha = 0.95$ is the empirically determined scaling exponent

The following cell implements this formula as a Python function.

In [ ]:
import numpy as np
import pandas as pd

def interestingness_score(count: int, total: int, alpha: float = 0.95) -> float:
    """
    Compute the interestingness score I(p, e) as defined in Section 2
    of Goorha & Ungar (2010).
    
    Parameters
    ----------
    count : int
        Number of co-occurrences of phrase p with entity e.
    total : int
        Total number of all phrase co-occurrences with entity e.
    alpha : float, default=0.95
        Power-law scaling exponent (Section 2, Equation 1).
    
    Returns
    -------
    float
        The interestingness score.
    """
    if total == 0:
        return 0.0
    return count / (total ** alpha)

# Quick sanity check
print(f"interestingness_score(10, 100, 0.95) = {interestingness_score(10, 100, 0.95):.6f}")
print(f"interestingness_score(1, 1, 0.95)    = {interestingness_score(1, 1, 0.95):.6f}")

## Section 2 Reference: Building the Toy Corpus

Section 2 describes applying the interestingness score to identify emerging entity-phrase associations. We replicate this using our **110-sample toy dataset** from Task 2.1, simulating iPod mentions within an IT Industry corpus. The dataset includes:
- iPod-related phrases (emerging signal)
- Generic IT phrases (stable noise)
- Rare junk phrases (sparsity test)

The cell below constructs the dataset and computes per-phrase totals for the current period (weeks 5–6).

In [ ]:
np.random.seed(42)

# ── Toy Dataset (same as Task 2.1) ───────────────────────────────────────────
ipod_phrases = {
    'iPod nano release':     {'baseline': [2, 3, 2, 3], 'current': [12, 18]},
    'iPod sales surge':      {'baseline': [1, 1, 2, 1], 'current': [9, 14]},
    'iPod vs Zune':          {'baseline': [0, 1, 1, 0], 'current': [7, 11]},
    'Apple iPod market':     {'baseline': [1, 2, 1, 2], 'current': [8, 13]},
    'new iPod features':     {'baseline': [0, 0, 1, 1], 'current': [6, 10]},
}

generic_phrases = {
    'technology industry':   {'baseline': [45, 48, 44, 50], 'current': [47, 49]},
    'software development':  {'baseline': [38, 40, 37, 42], 'current': [39, 41]},
    'data management':       {'baseline': [30, 32, 28, 35], 'current': [31, 33]},
    'cloud computing':       {'baseline': [25, 27, 24, 29], 'current': [26, 28]},
    'enterprise solutions':  {'baseline': [20, 22, 19, 23], 'current': [21, 22]},
    'network security':      {'baseline': [18, 20, 17, 21], 'current': [19, 20]},
    'digital transformation':{'baseline': [15, 17, 14, 18], 'current': [16, 17]},
    'IT infrastructure':     {'baseline': [22, 24, 21, 25], 'current': [23, 24]},
    'server management':     {'baseline': [12, 14, 11, 15], 'current': [13, 14]},
    'database optimization': {'baseline': [10, 12,  9, 13], 'current': [11, 12]},
}

junk_phrases = {
    'xyz_glitch_report':     {'baseline': [0, 0, 1, 0], 'current': [0, 0]},
    'q7_typo_artifact':      {'baseline': [0, 1, 0, 0], 'current': [0, 0]},
    'misc_fragment_99':      {'baseline': [1, 0, 0, 0], 'current': [0, 0]},
    'unknown_ref_alpha':     {'baseline': [0, 0, 0, 1], 'current': [0, 0]},
    'test_string_beta':      {'baseline': [0, 0, 0, 0], 'current': [1, 0]},
}

# Merge and compute current-period counts
all_phrases = {}
all_phrases.update(ipod_phrases)
all_phrases.update(generic_phrases)
all_phrases.update(junk_phrases)

records = []
for phrase, data in all_phrases.items():
    current_count = sum(data['current'])
    baseline_avg  = np.mean(data['baseline'])
    cat = 'iPod' if phrase in ipod_phrases else ('junk' if phrase in junk_phrases else 'generic')
    records.append({
        'phrase': phrase,
        'current_count': current_count,
        'baseline_avg': round(baseline_avg, 2),
        'category': cat
    })

df = pd.DataFrame(records)
total_current = df['current_count'].sum()
print(f"Total current-period counts: {total_current}")
print(f"\nDataset preview:")
print(df.to_string(index=False))

## Section 2 Reference: Computing Interestingness Scores

Following Section 2's methodology, we now apply the interestingness score $I(p, e) = \text{count}(p, e) / \text{total}(e)^{0.95}$ to every phrase in the current period. The `total` denominator is the sum of all current-period counts across the corpus, representing the overall "activity level" of the entity.

In [ ]:
# ── Apply Interestingness Score (Section 2, α = 0.95) ────────────────────────
alpha = 0.95

df['interestingness'] = df['current_count'].apply(
    lambda c: interestingness_score(c, total_current, alpha)
)

# Sort by score
df_ranked = df.sort_values('interestingness', ascending=False).reset_index(drop=True)
df_ranked.index += 1  # 1-indexed ranks
df_ranked.index.name = 'rank'

print("=== Interestingness Score Rankings (α = 0.95) ===")
print(f"Total current-period count (denominator): {total_current}")
print(f"total^0.95 = {total_current**0.95:.4f}\n")
print(df_ranked[['phrase', 'category', 'current_count', 'interestingness']].to_string())

## Section 2 Reference: Exploding Trend Filter

Section 2 also describes the *exploding trend* filter: a phrase is flagged only if its current count exceeds the baseline average by **≥ 50%**. We apply this filter below to identify which phrases qualify as "exploding trends" before scoring.

In [ ]:
# ── Exploding Trend Detection (Section 2) ────────────────────────────────────
THRESHOLD = 0.50  # 50% increase

df_ranked['pct_increase'] = df_ranked.apply(
    lambda row: ((row['current_count'] - row['baseline_avg']) / row['baseline_avg'] * 100)
                 if row['baseline_avg'] > 0 else float('inf'),
    axis=1
)

df_ranked['is_exploding'] = df_ranked['pct_increase'] >= (THRESHOLD * 100)

print("=== Exploding Trend Detection (≥50% increase over baseline) ===")
print(df_ranked[['phrase', 'category', 'baseline_avg', 'current_count',
                  'pct_increase', 'is_exploding']].to_string())

exploding = df_ranked[df_ranked['is_exploding']]
print(f"\n✅ {len(exploding)} phrases flagged as exploding trends:")
for _, row in exploding.iterrows():
    print(f"   • {row['phrase']} (score={row['interestingness']:.6f}, +{row['pct_increase']:.0f}%)")

## Summary

The interestingness score implementation follows Section 2 of the paper exactly:
1. **Formula:** $I(p, e) = \text{count}(p, e) / \text{total}(e)^{0.95}$
2. **Exploding trend filter:** ≥ 50% increase over 3-week (here 4-week) baseline
3. **Result:** iPod-related phrases are correctly identified as emerging trends and ranked above high-frequency generic noise.

The scores and rankings are saved for use in Task 2.3's bubble chart comparison.